# Thử nghiệm SLQ Spaced Layers (4 Layers cách đều: 6, 12, 18, 24)

**Mục tiêu:** So sánh với việc lấy 4 layers cuối (-4, -3, -2, -1). Mạng InternVL3-1B (Qwen2-0.5B backbone) có 24 layers. Lấy cách đều [6, 12, 18, 24] giúp thu thập đặc trưng từ các tầng nông (cú pháp, thực thể), trung gian (ngữ nghĩa tổng quát) đến tầng sâu (suy luận logic).

**Session options:** GPU T4x2 hoặc P100, Internet = On, Dataset = awsaf49/coco-2017-dataset

In [ ]:
!pip install -q "transformers==4.51.3" timm einops sentencepiece
import torch; print('GPU:', torch.cuda.get_device_name(0))

## 0. Kiểm tra dữ liệu COCO đã gắn đúng chưa

In [ ]:
import glob
print(glob.glob('/kaggle/input/**/captions_train2017.json', recursive=True))
print(glob.glob('/kaggle/input/**/train2017', recursive=True)[:3])

## 2. Huấn luyện SLQ trên COCO train (chỉ học 20 query)
Script tự tìm COCO trong `/kaggle/input`. Trên T4/P100 mô hình chạy **fp32** (fp16 gây NaN), nên chậm hơn: `--max_pairs 20000` mất khoảng 1,5–3 giờ. Nên dùng **Save Version → Save & Run All**. Gặp lỗi `CUDA out of memory` thì đổi `--batch_size` thành 32.

In [ ]:
!python slq_multilayer.py train --coco_root /kaggle/input --max_pairs 20000 --epochs 1 --batch_size 64 --layers 6,12,18,24 --head_type mlp --out slq_spaced_q20.pt

## 3. Đánh giá sau huấn luyện trên Flickr30K-1K và COCO-5K

In [ ]:
!python slq_multilayer.py eval --data flickr,coco --queries slq_spaced_q20.pt --layers 6,12,18,24 --head_type mlp --batch_size 32

## 4. (Tuỳ chọn) Dataset riêng
Upload một CSV gồm 2 cột `image,caption` cùng thư mục ảnh lên Kaggle Dataset, rồi bỏ dấu `#` ở dòng dưới:

In [ ]:
# !python slq.py eval --data custom --custom_csv /kaggle/input/<ten-dataset>/data.csv --image_root /kaggle/input/<ten-dataset>/images --queries slq_q20.pt

## Kết quả

In [ ]:
!cat results_*.json